# Project: Labels vs. Scale: Breaking Even on Banking77 with Encoders and Decoders
#### Evaluation

### Evaluation

This notebook scores every model's predictions the **same way**, so the encoder, the decoder, the baselines and the frontier control are compared on equal terms.

Every model writes its predictions to a CSV that follows `results/PREDICTION_FORMAT.md`. This notebook reads those files, **checks them against the format first**, and only then scores them.

**Design**

| Piece | What it is | Why |
|---|---|---|
| Format rules | The allowed columns, models, budgets, seeds and splits, written as Python constants | One place to check every file against; must match `results/PREDICTION_FORMAT.md` |
| Loader | `load_predictions(path)` reads one prediction CSV and runs every format check | A broken file is caught **before** it produces a wrong number |
| Metrics | Accuracy (primary) and macro-F1 (secondary) | Accuracy is what prior Banking77 work reports; macro-F1 gives every intent equal weight |
| Per-intent view | Precision, recall and F1 per intent, plus a confusion matrix | Shows **which** intents fail and which ones each model confuses |
| Seed summary | Mean and standard deviation across seeds 0, 1, 2 | Separates a real difference from luck in which examples were sampled |
| Learning curves | Accuracy against budget, one line per model | Where the lines cross answers the research question |

**Ground rules**

- Tune on `val`, report on `test`. Test predictions are used only for final numbers.
- A decoder answer that matches no intent is stored as `-1` and **counted as wrong**, never dropped. Dropping it would inflate the decoder's accuracy.

#### Imports

In [ ]:
from pathlib import Path

import pandas as pd

# Find the project root (the folder that contains scripts/download_data.py), same as data_splits.ipynb
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / "scripts" / "download_data.py").exists()), cwd)
RESULTS = ROOT / "results"   # prediction CSVs live here

#### Format rules

These constants are `results/PREDICTION_FORMAT.md` written as code. **If the format changes, change it in both places.**

| Constant | What it holds |
|---|---|
| `REQUIRED_COLUMNS` | Every column a prediction file must have |
| `MODEL_TYPE` | Each allowed model name and its type (`encoder`, `decoder`, `baseline`) |
| `BUDGETS` | Every allowed training-data size, from `zeroshot` to `full` |
| `SEEDS` | The three seeds |
| `SPLIT_SIZES` | How many rows a complete file must have for each split |
| `UNMATCHED` | The value a decoder prediction gets when its text matches no intent |

In [ ]:
REQUIRED_COLUMNS = [
    "model", "type", "budget", "seed", "split",
    "example_id", "true_label", "predicted_label", "raw_output",
]

MODEL_TYPE = {
    "modernbert": "encoder",
    "qwen_zeroshot": "decoder",
    "qwen_qlora": "decoder",
    "frontier": "decoder",
    "tfidf_lr": "baseline",
    "majority": "baseline",
}

BUDGETS = ["zeroshot", "k1", "k3", "k5", "k10", "k20", "k30", "p50", "full"]
SEEDS = [0, 1, 2]
SPLIT_SIZES = {"val": 385, "test": 3080}   # rows every complete file must have
N_LABELS = 77                               # -1 for wrong labels are not counted
UNMATCHED = -1                              # decoder output that matched no intent

#### Load and check one prediction file

`load_predictions(path)` reads one CSV and runs six checks **before** anything is scored. If a check fails, it lists **every** problem it found at once, so a broken file can be fixed in one pass.

| # | Check | What it catches |
|---|---|---|
| 1 | All required columns are present | A run that forgot `split` or `raw_output` |
| 2 | One file holds one run | Two seeds or two budgets saved into the same file |
| 3 | Every value is an allowed one | Typos like `qwen_Qlora`; a `type` that does not match its `model` |
| 4 | The file name matches the columns inside | A file named `k5` that actually holds `k10` predictions |
| 5 | Labels are in range | A conversion bug producing label `77` or `-5`; `-1` is allowed only in `predicted_label` |
| 6 | Every example appears exactly once | Duplicates, or a run that crashed partway (for example 2,000 of 3,080 test rows) |

**Why check 6 matters most:** if one model's run stopped early and nobody noticed, its accuracy would be computed on fewer questions than the other model's, and the comparison would be wrong without any error showing.

The function only **checks**; scoring comes next.

In [ ]:
def load_predictions(path):
    """Read one prediction CSV and check it follows the format.

    Returns the DataFrame if every check passes. Otherwise raises a ValueError
    that lists every problem found.
    """
    path = Path(path)
    df = pd.read_csv(path, keep_default_na=False)   # keep an empty raw_output as "", not NaN
    problems = []

    # 1. All required columns are present (stop here if not: the other checks need them)
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing}")

    # 2. One file = one run: a single model, type, budget, seed and split
    for col in ["model", "type", "budget", "seed", "split"]:
        if df[col].nunique() != 1:
            problems.append(f"'{col}' should have one value per file, found {sorted(df[col].unique())}")
    if problems:
        raise ValueError(f"{path.name}:\n  - " + "\n  - ".join(problems))

    model, mtype, budget = df["model"].iloc[0], df["type"].iloc[0], df["budget"].iloc[0]
    seed, split = int(df["seed"].iloc[0]), df["split"].iloc[0]

    # 3. Each value is one of the allowed ones
    if model not in MODEL_TYPE:
        problems.append(f"unknown model '{model}'")
    elif mtype != MODEL_TYPE[model]:
        problems.append(f"type '{mtype}' does not match model '{model}' (expected '{MODEL_TYPE[model]}')")
    if budget not in BUDGETS:
        problems.append(f"unknown budget '{budget}'")
    if seed not in SEEDS:
        problems.append(f"seed {seed} not in {SEEDS}")
    if split not in SPLIT_SIZES:
        problems.append(f"unknown split '{split}'")

    # 4. The file name matches the columns inside it
    expected_name = f"{model}_{budget}_seed{seed}_{split}.csv"
    if path.name != expected_name:
        problems.append(f"file name should be '{expected_name}'")

    # 5. Labels are in range (-1 is allowed only as a prediction)
    if not df["true_label"].between(0, N_LABELS - 1).all():
        problems.append("true_label has values outside 0-76")
    pred_ok = df["predicted_label"].between(0, N_LABELS - 1) | (df["predicted_label"] == UNMATCHED)
    if not pred_ok.all():
        problems.append("predicted_label has values outside 0-76 (or -1)")

    # 6. Every example appears exactly once, and none are missing
    if not df["example_id"].is_unique:
        problems.append("example_id has duplicates")
    if split in SPLIT_SIZES and len(df) != SPLIT_SIZES[split]:
        problems.append(f"{split} file should have {SPLIT_SIZES[split]} rows, found {len(df)}")

    if problems:
        raise ValueError(f"{path.name}:\n  - " + "\n  - ".join(problems))
    return df

#### Accuracy and macro-F1

`score(df)` takes one loaded prediction file and returns its two headline numbers.

| Metric | What it measures | Why we use it |
|---|---|---|
| Accuracy (primary) | Share of examples predicted correctly | What prior Banking77 work reports; fair here because the test set is balanced |
| Macro-F1 (secondary) | F1 for each intent, then the plain average | To flag when some labels are underperforming, even if overall accuracy looks fine

**How `-1` is handled:** an unmatched decoder answer never equals a true label, so it is automatically counted as **wrong** in both metrics. For macro-F1 we pass `labels=range(77)` so that `-1` is not treated as a 78th intent

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def score(df):
    """Accuracy and macro-F1 for one prediction file."""
    y_true, y_pred = df["true_label"], df["predicted_label"]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=list(range(N_LABELS)), average="macro", zero_division=0),
    }

#### Per-intent scores and confusion matrix

Macro-F1 tells us **that** some intents are underperforming. These two functions show **which** ones, and **what they are mistaken for**.

| Function | Returns | Use it to |
|---|---|---|
| `per_intent(df)` | One row per intent: precision, recall, F1, and how many test examples it has | Sort by F1 to find the weakest intents |
| `top_confusions(df, n)` | The `n` most frequent mistakes as (true intent → predicted intent, count) | See which near-duplicate intents a model mixes up, and compare models (Task 10.2) |
| `confusion(df)` | The full table: rows are true intents, columns are predicted intents | Plotting, or any custom analysis |

Unmatched decoder answers appear as their own column, `unmatched`, so they are visible rather than hidden.

The full 77 × 77 matrix is too large to read by eye, which is why `top_confusions` exists: it lists only the mistakes that actually happen most.

In [ ]:
import json
from sklearn.metrics import precision_recall_fscore_support

LABEL_NAMES = json.load(open(ROOT / "data" / "raw" / "label_names.json"))   # label id -> intent name
NAMES = {i: name for i, name in enumerate(LABEL_NAMES)}
NAMES[UNMATCHED] = "unmatched"

def per_intent(df):
    """Precision, recall, F1 and support for each of the 77 intents."""
    p, r, f, n = precision_recall_fscore_support(
        df["true_label"], df["predicted_label"], labels=list(range(N_LABELS)), zero_division=0)
    return pd.DataFrame({"intent": LABEL_NAMES, "precision": p, "recall": r, "f1": f, "support": n})

def confusion(df):
    """Full confusion matrix: rows = true intent, columns = predicted intent (incl. 'unmatched')."""
    return pd.crosstab(df["true_label"].map(NAMES), df["predicted_label"].map(NAMES),
                       rownames=["true"], colnames=["predicted"])

def top_confusions(df, n=10):
    """The n most frequent mistakes as (true -> predicted, count)."""
    wrong = df[df["true_label"] != df["predicted_label"]]
    return (wrong.groupby(["true_label", "predicted_label"]).size()
                 .sort_values(ascending=False).head(n)
                 .rename("count").reset_index()
                 .assign(true=lambda d: d["true_label"].map(NAMES),
                         predicted=lambda d: d["predicted_label"].map(NAMES))
                 [["true", "predicted", "count"]])

#### Seed summary and learning curves

Every run is one file. These functions combine all of them into the results that answer the research question.

| Function | Returns | Why |
|---|---|---|
| `score_all(split)` | One row per file in `results/`: model, budget, seed, accuracy, macro-F1 | Scores every run in one call |
| `summarize(scores)` | Mean and standard deviation across seeds, for each model and budget | The mean is the result and the spread shows how much it depends on **which** examples were sampled |
| `plot_learning_curve(summary, metric)` | Score against training size, one line per model | Where the encoder and decoder lines cross is the break-even point |

In [ ]:
import matplotlib.pyplot as plt

# Number of training examples behind each budget (from data/splits/manifest.json)
BUDGET_SIZE = {"k1": 77, "k3": 231, "k5": 385, "k10": 770, "k20": 1540, "k30": 2310, "p50": 4807, "full": 9618}

def score_all(split="test"):
    """Load and score every prediction file in results/ for one split."""
    rows = []
    for path in sorted(RESULTS.glob(f"*_{split}.csv")):
        df = load_predictions(path)
        info = df.iloc[0][["model", "type", "budget", "seed", "split"]].to_dict()
        rows.append({**info, **score(df)})
    return pd.DataFrame(rows)

def summarize(scores):
    """Mean and standard deviation across seeds for each model and budget."""
    return (scores.groupby(["model", "type", "budget"])
                  .agg(accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
                       macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
                       n_seeds=("seed", "nunique"))
                  .reset_index())

def plot_learning_curve(summary, metric="accuracy"):
    """Score against number of training examples, one line per model."""
    fig, ax = plt.subplots(figsize=(8, 5))
    for model, g in summary.groupby("model"):
        if (g["budget"] == "zeroshot").all():          # no training data: a flat reference line
            ax.axhline(g[f"{metric}_mean"].iloc[0], linestyle="--", label=model)
            continue
        g = g.assign(n=g["budget"].map(BUDGET_SIZE)).dropna(subset=["n"]).sort_values("n")
        mean, std = g[f"{metric}_mean"], g[f"{metric}_std"].fillna(0)
        ax.plot(g["n"], mean, marker="o", label=model)
        ax.fill_between(g["n"], mean - std, mean + std, alpha=0.2)
    ax.set_xscale("log")
    ax.set_xticks(list(BUDGET_SIZE.values()), list(BUDGET_SIZE.keys()))
    ax.set_xlabel("Training budget (log scale)")
    ax.set_ylabel(metric.replace("_", "-"))
    ax.set_title(f"Learning curves: {metric.replace('_', '-')} on test")
    ax.legend()
    plt.show()